### Churn Modeling pdf extract

In [0]:
%pip install pdfplumber --quiet
dbutils.library.restartPython()

In [0]:
import pdfplumber
import re

PDF_PATH = "/Volumes/customer_360/source_data/raw/Churn_modeling_pdf/Churn_Modelling.pdf"

# Handles rows like:
# 420 15615624De Salis 605France Female 28 6
BLOCK1_PATTERN = re.compile(
    r'^(\d+)\s+'                    # row_number
    r'(\d{8})'                      # customer_id
    r'(.+?)\s+'                     # surname
    r'(\d{3})'                      # credit_score
    r'(France|Germany|Spain)\s+'    # geography
    r'(Male|Female)\s+'             # gender
    r'(\d+)\s+'                     # age
    r'(\d+)$'                       # tenure
)

block1_rows = []
parse_failures_1 = []

with pdfplumber.open(PDF_PATH) as pdf:

    total_pages = len(pdf.pages)
    block1_page_count = total_pages // 2

    for page_idx in range(block1_page_count):

        page = pdf.pages[page_idx]
        text = page.extract_text()

        if not text:
            continue

        for line in text.split("\n"):

            line = line.strip()

            if (
                not line
                or line.startswith("Churn_Modelling")
                or line.startswith("Page")
                or line.startswith("RowNumber")
            ):
                continue

            match = BLOCK1_PATTERN.match(line)

            if match:

                block1_rows.append({
                    "row_number": int(match.group(1)),
                    "customer_id": match.group(2),
                    "surname": match.group(3).strip(),
                    "credit_score": int(match.group(4)),
                    "geography": match.group(5),
                    "gender": match.group(6),
                    "age": int(match.group(7)),
                    "tenure": int(match.group(8))
                })

            else:
                parse_failures_1.append(line)

print(f"Total pages           : {total_pages}")
print(f"Block1 pages          : {block1_page_count}")
print(f"Block1 rows parsed    : {len(block1_rows)}")
print(f"Block1 failures       : {len(parse_failures_1)}")

if parse_failures_1:
    print("\nFirst 20 failures:")
    for x in parse_failures_1[:20]:
        print(repr(x))

In [0]:
BLOCK2_PATTERN = re.compile(
    r'^([\d.]+)\s(\d+)\s([01])\s([01])\s([\d.]+)\s([01])$'
)
 
block2_rows = []
parse_failures_2 = []
 
with pdfplumber.open(PDF_PATH) as pdf:
    for page_idx in range(block1_page_count, total_pages):
        text = pdf.pages[page_idx].extract_text()
        if not text:
            continue
        for line in text.split("\n"):
            line = line.strip()
            if not line or line.startswith("Churn_Modelling") or line.startswith("Page") \
               or line.startswith("Balance"):
                continue
            m = BLOCK2_PATTERN.match(line)
            if m:
                block2_rows.append({
                    "balance":            float(m.group(1)),
                    "num_of_products":    int(m.group(2)),
                    "has_cr_card":        int(m.group(3)),
                    "is_active_member":   int(m.group(4)),
                    "estimated_salary":   float(m.group(5)),
                    "exited":             int(m.group(6)),
                })
            else:
                parse_failures_2.append(line)
 
print(f"Block 2 rows parsed       : {len(block2_rows):,}")
print(f"Block 2 parse failures    : {len(parse_failures_2)}")
if parse_failures_2[:5]:
    print("Sample failures:", parse_failures_2[:5])
 

### zip Block 1 + Block 2 by row order

In [0]:
assert len(block1_rows) == len(block2_rows), (
    f"Row count mismatch! Block 1: {len(block1_rows)} vs Block 2: {len(block2_rows)} "
    f"-- check parse_failures before proceeding."
)
 
merged_rows = []
for b1, b2 in zip(block1_rows, block2_rows):
    merged = {**b1, **b2}
    merged_rows.append(merged)
 
print(f"Merged rows               : {len(merged_rows):,}")
print(f"Expected                  : 10,000")
print()
print("Sample merged row (row_number=1):")
for r in merged_rows:
    if r["row_number"] == 1:
        print(r)
        break

In [0]:
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, IntegerType, StringType, DoubleType
)
 
pdf_pandas = pd.DataFrame(merged_rows)
 
schema = StructType([
    StructField("row_number",        IntegerType(), True),
    StructField("customer_id",       StringType(),  True),
    StructField("surname",           StringType(),  True),
    StructField("credit_score",      IntegerType(), True),
    StructField("geography",         StringType(),  True),
    StructField("gender",            StringType(),  True),
    StructField("age",               IntegerType(), True),
    StructField("tenure",            IntegerType(), True),
    StructField("balance",           DoubleType(),  True),
    StructField("num_of_products",   IntegerType(), True),
    StructField("has_cr_card",       IntegerType(), True),
    StructField("is_active_member",  IntegerType(), True),
    StructField("estimated_salary",  DoubleType(),  True),
    StructField("exited",            IntegerType(), True),
])
 
spark_df = spark.createDataFrame(pdf_pandas, schema=schema)
 
spark_df = (
    spark_df
    .withColumn("source_file", F.lit("Churn_Modelling.pdf"))
    .withColumn("ingestion_time", F.current_timestamp())
)
 
(
    spark_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("customer_360.bronze.bronze_churn_benchmark")
)
 
print("bronze_churn_benchmark written successfully ✓")
 

In [0]:
%sql
SELECT
  COUNT(*)                                AS total_rows,        
  COUNT(DISTINCT customer_id)              AS unique_customers,  
  COUNT(CASE WHEN exited = 1 THEN 1 END)   AS churned_customers,
  ROUND(AVG(exited) * 100, 2)               AS churn_rate_pct,
  MIN(credit_score)                         AS min_credit_score,
  MAX(credit_score)                         AS max_credit_score,
  COUNT(DISTINCT geography)                 AS unique_countries
FROM customer_360.bronze.bronze_churn_benchmark;